# LeetCode 1481 — Least Number of Unique Integers after K Removals bằng Python

Link đề: https://leetcode.com/problems/least-number-of-unique-integers-after-k-removals/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Greedy, Sorting, Counting

Skeleton LeetCode:

```python
class Solution:
    def findLeastNumOfUniqueInts(self, arr: List[int], k: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Chuyển bài toán về **tần suất**: chỉ số lần xuất hiện của mỗi giá trị là quan trọng, không phải giá trị cụ thể.
2. Hiểu chiến lược **tham lam**: muốn "xóa sổ" được nhiều giá trị nhất với `k` lần xóa thì xóa các giá trị **ít xuất hiện nhất** trước — và chứng minh được vì sao đúng.
3. Dùng `collections.Counter`, sắp xếp danh sách tần suất, `heapq`, và **counting sort** trên tần suất để đạt `O(n)`.
4. Xử lý đúng phần dư: khi `k` không đủ xóa hết một giá trị thì **dừng**, phần còn lại của `k` xóa lặt vặt không làm giảm số giá trị phân biệt.
5. Tránh lỗi: xóa giá trị nhiều nhất trước, so sánh `>` thay cho `>=`, trừ `k` âm, nhầm "xóa k giá trị nhỏ nhất".

## 2. Đề bài & ví dụ

Cho mảng số nguyên `arr` và số nguyên `k`. Hãy **xóa đúng `k` phần tử** sao cho số lượng **giá trị phân biệt** còn lại là **ít nhất**. Trả về số lượng đó.

**Ví dụ 1:** `arr = [5,5,4], k = 1` → `1` (xóa số `4`, còn `5`).

**Ví dụ 2:** `arr = [4,3,1,1,3,3,2], k = 3` → `2` (xóa `4`, `2` và một trong hai số `1` hoặc một số `3`; còn lại `1` và `3`).

**Ràng buộc:**
- `1 <= arr.length <= 10^5`
- `1 <= arr[i] <= 10^9`
- `0 <= k <= arr.length`

Lưu ý: `k = arr.length` ⇒ xóa hết ⇒ đáp án `0`; `k = 0` ⇒ đáp án là số giá trị phân biệt ban đầu.

In [1]:
import heapq
import random
import time
from collections import Counter
from itertools import combinations
from typing import List

examples = [
    ([5, 5, 4], 1, 1),
    ([4, 3, 1, 1, 3, 3, 2], 3, 2),
]
for arr, k, want in examples:
    print(f"arr={arr}, k={k} -> kỳ vọng {want}")

arr=[5, 5, 4], k=1 -> kỳ vọng 1
arr=[4, 3, 1, 1, 3, 3, 2], k=3 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1 `collections.Counter` — đếm tần suất

`Counter(arr)` trả về dict `giá trị → số lần xuất hiện`. `.values()` cho danh sách tần suất; `len(counter)` là số giá trị phân biệt.

In [2]:
cnt = Counter([4, 3, 1, 1, 3, 3, 2])
print(cnt)
print("tần suất:", list(cnt.values()), "| số giá trị phân biệt:", len(cnt))
print("most_common:", cnt.most_common())

Counter({3: 3, 1: 2, 4: 1, 2: 1})
tần suất: [1, 3, 2, 1] | số giá trị phân biệt: 4
most_common: [(3, 3), (1, 2), (4, 1), (2, 1)]


### 3.2 Sắp xếp danh sách tần suất

`sorted(cnt.values())` cho các tần suất tăng dần — thứ tự mà ta sẽ "xóa sổ" các giá trị.

In [3]:
freqs = sorted(cnt.values())
print(freqs)

[1, 1, 2, 3]


### 3.3 `heapq` — hàng đợi ưu tiên (min-heap)

`heapq.heapify(lst)` biến list thành min-heap trong `O(n)`; `heappop` lấy phần tử nhỏ nhất `O(log n)`. Dùng khi chỉ cần lấy dần vài phần tử nhỏ nhất mà không muốn sắp toàn bộ.

In [4]:
h = [3, 1, 2, 1]
heapq.heapify(h)
print([heapq.heappop(h) for _ in range(4)])

[1, 1, 2, 3]


### 3.4 Đếm "tần suất của tần suất" (counting sort)

Tần suất nằm trong `[1, n]`. Tạo mảng `fc` độ dài `n + 1`, `fc[f]` = số giá trị có tần suất đúng bằng `f`. Duyệt `f` từ nhỏ tới lớn tương đương duyệt các tần suất đã sắp — không cần `sort`, `O(n)`.

In [5]:
arr = [4, 3, 1, 1, 3, 3, 2]
fc = [0] * (len(arr) + 1)
for f in Counter(arr).values():
    fc[f] += 1
print("fc =", fc, "-> có", fc[1], "giá trị xuất hiện 1 lần,", fc[2], "giá trị 2 lần,", fc[3], "giá trị 3 lần")

fc = [0, 2, 1, 1, 0, 0, 0, 0] -> có 2 giá trị xuất hiện 1 lần, 1 giá trị 2 lần, 1 giá trị 3 lần


### 3.5 `itertools.combinations` — thử mọi cách chọn chỉ số

`combinations(range(n), k)` sinh mọi tập `k` chỉ số — dùng cho brute force "thử mọi cách xóa k phần tử" (chỉ khả thi khi `n` nhỏ).

In [6]:
print(list(combinations(range(4), 2)))

[(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 2: `arr = [4,3,1,1,3,3,2]`, `k = 3`. Đếm: `4:1, 2:1, 1:2, 3:3`.

Muốn giảm số giá trị phân biệt, ta phải **xóa sạch mọi bản sao** của một giá trị. Xóa sạch giá trị có tần suất `f` tốn `f` lần xóa. Với ngân sách `k = 3`:

| thứ tự thử (tần suất tăng) | tần suất | k còn lại trước | xóa sạch được? | k sau | số giá trị còn |
|---|---|---|---|---|---|
| giá trị 4 | 1 | 3 | ✔ | 2 | 3 |
| giá trị 2 | 1 | 2 | ✔ | 1 | 2 |
| giá trị 1 | 2 | 1 | ✘ (cần 2) | dừng | **2** |

Lượt xóa còn dư (`k = 1`) xóa bớt một bản sao bất kỳ — không làm mất giá trị nào. Đáp án `2`.

Trực giác: giống như "mua được nhiều món nhất với số tiền cố định" ⇒ mua **món rẻ nhất trước**.

In [7]:
arr, k = [4, 3, 1, 1, 3, 3, 2], 3
freqs = sorted(Counter(arr).values())
left = len(freqs)
for f in freqs:
    if k >= f:
        k -= f
        left -= 1
        print(f"xóa sạch giá trị có tần suất {f}: k còn {k}, còn {left} giá trị")
    else:
        print(f"tần suất {f} > k = {k}: dừng")
        break
print("đáp án:", left)

xóa sạch giá trị có tần suất 1: k còn 2, còn 3 giá trị
xóa sạch giá trị có tần suất 1: k còn 1, còn 2 giá trị
tần suất 2 > k = 1: dừng
đáp án: 2


## 5. Cách 1 — Brute Force / baseline: thử mọi cách xóa k phần tử

**Ý tưởng:** với mọi tập `k` chỉ số (`C(n, k)` tập), xóa chúng và đếm số giá trị phân biệt còn lại; lấy nhỏ nhất. Không dựa vào suy luận nào ⇒ chuẩn đối chiếu đáng tin.

In [8]:
def least_unique_brute(arr, k):
    n = len(arr)
    best = len(set(arr))
    for removed in combinations(range(n), k):
        rem = set(removed)
        left = len({arr[i] for i in range(n) if i not in rem})
        best = min(best, left)
    return best

for arr, k, want in examples:
    got = least_unique_brute(arr, k)
    print(got, want, "✔" if got == want else "✘")

1 1 ✔
2 2 ✔


**Độ phức tạp:** `O(C(n, k) · n)` — bùng nổ tổ hợp. `n = 20, k = 10` đã có `184756` tập.

**Nút thắt:** rất nhiều cách xóa là "tương đương" (xóa bản sao nào của cùng giá trị cũng như nhau), và việc xóa lẻ tẻ không hoàn toàn một giá trị là **lãng phí**. Chỉ cần quyết định: **xóa sạch những giá trị nào**.

In [9]:
rng = random.Random(1481)
for n in (12, 16, 18):
    arr = [rng.randint(1, 5) for _ in range(n)]
    t = time.perf_counter()
    r = least_unique_brute(arr, n // 2)
    print(f"n={n}, k={n // 2}: đáp án={r}, {time.perf_counter() - t:.3f}s")

n=12, k=6: đáp án=2, 0.001s
n=16, k=8: đáp án=2, 0.019s
n=18, k=9: đáp án=2, 0.043s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ tần suất quan trọng:** đổi `arr` thành danh sách tần suất `f_1, f_2, …, f_d` (`d` giá trị phân biệt). Đáp án = `d − (số giá trị bị xóa sạch)`.

**Insight 2 — tham lam:** muốn xóa sạch được **nhiều** giá trị nhất với tổng chi phí `≤ k` ⇒ chọn các giá trị có tần suất **nhỏ nhất**.

**Quy tắc:**
- Sắp tần suất tăng dần. **Nếu** `k >= f` **thì** xóa sạch: `k -= f`, `removed += 1`.
- **Ngược lại** dừng (các tần suất sau còn lớn hơn, không xóa sạch được).
- Số lượt xóa còn dư dùng để xóa bớt bản sao — không ảnh hưởng đáp án (luôn có đủ phần tử để xóa vì `k ≤ n`).

**Invariant:** sau khi xét `t` tần suất nhỏ nhất, `removed` là số lớn nhất các giá trị có thể xóa sạch với số lượt đã dùng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `Counter` + sắp tần suất + tham lam

**Từng đoạn:**
1. `freqs = sorted(Counter(arr).values())`.
2. Duyệt `f` trong `freqs`: nếu `k >= f` thì `k -= f`, ngược lại `break` và trả về số tần suất còn lại.

In [10]:
def least_unique_sort(arr, k):
    freqs = sorted(Counter(arr).values())
    for i, f in enumerate(freqs):
        if k >= f:
            k -= f
        else:
            return len(freqs) - i            # các giá trị từ i trở đi còn lại
    return 0                                 # xóa sạch tất cả

for arr, k, want in examples:
    print(least_unique_sort(arr, k), want)

1 1
2 2


Code nộp LeetCode (Cách 2):

In [11]:
class Solution:
    def findLeastNumOfUniqueInts(self, arr: List[int], k: int) -> int:
        freqs = sorted(Counter(arr).values())
        for i, f in enumerate(freqs):
            if k < f:
                return len(freqs) - i
            k -= f
        return 0

print([Solution().findLeastNumOfUniqueInts(a, k) for a, k, _ in examples])

[1, 2]


**Độ phức tạp:** đếm `O(n)`, sắp `O(d log d)` với `d ≤ n` giá trị phân biệt ⇒ `O(n log n)` xấu nhất; bộ nhớ `O(d)`.

### 7.2 Cách 3 — Min-heap tần suất

**Từng đoạn:**
1. `h = list(Counter(arr).values())`, `heapify(h)`.
2. Trong khi `h` khác rỗng và `h[0] <= k`: `k -= heappop(h)`.
3. Trả về `len(h)`.

Có lợi khi `k` nhỏ (chỉ pop vài phần tử): `O(d + r log d)` với `r` là số giá trị bị xóa sạch.

In [12]:
def least_unique_heap(arr, k):
    h = list(Counter(arr).values())
    heapq.heapify(h)
    while h and h[0] <= k:
        k -= heapq.heappop(h)
    return len(h)

for arr, k, want in examples:
    print(least_unique_heap(arr, k), want)

1 1
2 2


Code nộp LeetCode (Cách 3):

In [13]:
class Solution:
    def findLeastNumOfUniqueInts(self, arr: List[int], k: int) -> int:
        h = list(Counter(arr).values())
        heapq.heapify(h)
        while h and h[0] <= k:
            k -= heapq.heappop(h)
        return len(h)

print([Solution().findLeastNumOfUniqueInts(a, k) for a, k, _ in examples])

[1, 2]


**Độ phức tạp:** `O(n + r log d)` thời gian, `O(d)` bộ nhớ.

### 7.3 Cách 4 — Counting sort trên tần suất (`O(n)`)

**Từng đoạn:**
1. `fc[f]` = số giá trị có tần suất `f` (`1 ≤ f ≤ n`).
2. Duyệt `f = 1, 2, …`: có `fc[f]` giá trị cùng chi phí `f`. Số xóa sạch được ở mức này: `take = min(fc[f], k // f)`.
3. `k -= take * f`, `removed += take`. Nếu `take < fc[f]` thì không thể xóa thêm ở mức này, và các mức sau còn đắt hơn ⇒ dừng.
4. Đáp án `d − removed`.

In [14]:
def least_unique_bucket(arr, k):
    cnt = Counter(arr)
    n = len(arr)
    fc = [0] * (n + 1)
    for f in cnt.values():
        fc[f] += 1
    removed = 0
    for f in range(1, n + 1):
        if fc[f] == 0:
            continue
        take = min(fc[f], k // f)
        k -= take * f
        removed += take
        if take < fc[f]:
            break
    return len(cnt) - removed

for arr, k, want in examples:
    print(least_unique_bucket(arr, k), want)

1 1
2 2


Code nộp LeetCode (Cách 4):

In [15]:
class Solution:
    def findLeastNumOfUniqueInts(self, arr: List[int], k: int) -> int:
        cnt = Counter(arr)
        fc = [0] * (len(arr) + 1)
        for f in cnt.values():
            fc[f] += 1
        removed = 0
        for f in range(1, len(arr) + 1):
            if fc[f]:
                take = min(fc[f], k // f)
                k -= take * f
                removed += take
                if take < fc[f]:
                    break
        return len(cnt) - removed

print([Solution().findLeastNumOfUniqueInts(a, k) for a, k, _ in examples])

[1, 2]


**Độ phức tạp:** `O(n)` thời gian và bộ nhớ. Trong Python, `sorted` (C) trên `d` phần tử thường đã rất nhanh nên Cách 2 và Cách 4 chênh nhau không nhiều.

In [16]:
rng = random.Random(3)
cases = {
    "nhiều giá trị (d≈n)": [rng.randint(1, 10**9) for _ in range(10**5)],
    "ít giá trị (d≈1000)": [rng.randint(1, 1000) for _ in range(10**5)],
}
for label, arr in cases.items():
    for name, f in [("sort", least_unique_sort), ("heap", least_unique_heap), ("bucket", least_unique_bucket)]:
        t = time.perf_counter()
        r = f(arr, len(arr) // 3)
        print(f"{label:20s} {name:6s}: {r:6d}  {time.perf_counter() - t:.3f}s")

nhiều giá trị (d≈n)  sort  :  66664  0.015s


nhiều giá trị (d≈n)  heap  :  66664  0.023s


nhiều giá trị (d≈n)  bucket:  66664  0.012s
ít giá trị (d≈1000)  sort  :    631  0.004s
ít giá trị (d≈1000)  heap  :    631  0.004s
ít giá trị (d≈1000)  bucket:    631  0.005s


## 8. TRACE từng bước

Trace Cách 2 với `arr = [2,4,1,8,3,5,1,3]`, `k = 3`. Đếm: `2:1, 4:1, 8:1, 5:1, 1:2, 3:2` ⇒ tần suất sắp: `[1,1,1,1,2,2]`, `d = 6`.

| i | f | k trước | k ≥ f? | k sau | còn lại |
|---|---|---|---|---|---|
| 0 | 1 | 3 | ✔ | 2 | 5 |
| 1 | 1 | 2 | ✔ | 1 | 4 |
| 2 | 1 | 1 | ✔ | 0 | 3 |
| 3 | 1 | 0 | ✘ → dừng | | **3** |

In [17]:
def trace_sort(arr, k):
    cnt = Counter(arr)
    freqs = sorted(cnt.values())
    print("Counter:", dict(cnt))
    print("tần suất sắp:", freqs, "| d =", len(freqs))
    for i, f in enumerate(freqs):
        if k >= f:
            k -= f
            print(f"  i={i} f={f}: xóa sạch -> k={k}, còn {len(freqs) - i - 1} giá trị")
        else:
            print(f"  i={i} f={f} > k={k}: dừng -> đáp án {len(freqs) - i}")
            return len(freqs) - i
    print("  xóa sạch tất cả -> đáp án 0")
    return 0

_ = trace_sort([2, 4, 1, 8, 3, 5, 1, 3], 3)

Counter: {2: 1, 4: 1, 1: 2, 8: 1, 3: 2, 5: 1}
tần suất sắp: [1, 1, 1, 1, 2, 2] | d = 6
  i=0 f=1: xóa sạch -> k=2, còn 5 giá trị
  i=1 f=1: xóa sạch -> k=1, còn 4 giá trị
  i=2 f=1: xóa sạch -> k=0, còn 3 giá trị
  i=3 f=1 > k=0: dừng -> đáp án 3


In [18]:
def trace_bucket(arr, k):
    cnt = Counter(arr)
    fc = [0] * (len(arr) + 1)
    for f in cnt.values():
        fc[f] += 1
    print("fc (chỉ in f có giá trị):", {f: c for f, c in enumerate(fc) if c})
    removed = 0
    for f in range(1, len(arr) + 1):
        if fc[f]:
            take = min(fc[f], k // f)
            k -= take * f
            removed += take
            print(f"  f={f}: có {fc[f]} giá trị, xóa sạch {take} -> k={k}, removed={removed}")
            if take < fc[f]:
                print("  không xóa hết mức này -> dừng")
                break
    print("đáp án:", len(cnt) - removed)
    return len(cnt) - removed

_ = trace_bucket([4, 3, 1, 1, 3, 3, 2], 3)

fc (chỉ in f có giá trị): {1: 2, 2: 1, 3: 1}
  f=1: có 2 giá trị, xóa sạch 2 -> k=1, removed=2
  f=2: có 1 giá trị, xóa sạch 0 -> k=1, removed=2
  không xóa hết mức này -> dừng
đáp án: 2


## 9. Code nộp LeetCode

Bản khuyên dùng (Counter + sắp tần suất), đúng chữ ký `findLeastNumOfUniqueInts(self, arr: List[int], k: int) -> int`:

In [19]:
class Solution:
    def findLeastNumOfUniqueInts(self, arr: List[int], k: int) -> int:
        freqs = sorted(Counter(arr).values())     # tần suất tăng dần
        for i, f in enumerate(freqs):
            if k < f:                             # không đủ để xóa sạch giá trị này
                return len(freqs) - i
            k -= f
        return 0

for a, k, want in examples:
    assert Solution().findLeastNumOfUniqueInts(a, k) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Chỉ xóa sạch mới có ích.** Số giá trị phân biệt chỉ giảm khi **mọi** bản sao của một giá trị bị xóa. Một lời giải tối ưu có thể biến đổi để mọi lượt xóa "dở dang" được dồn về cuối — chúng không ảnh hưởng đáp án. Vì `k ≤ n`, luôn có đủ phần tử để tiêu nốt lượt dư.
2. **Bài toán con:** chọn tập giá trị `S` để xóa sạch, `Σ_{v∈S} f_v ≤ k`, **tối đa hóa** `|S|`.
3. **Tham lam tối ưu (exchange argument):** giả sử lời giải tối ưu `S*` có `|S*| = r`. Thay mỗi phần tử của `S*` bằng một trong `r` tần suất nhỏ nhất — tổng chi phí không tăng (tổng `r` số nhỏ nhất ≤ tổng `r` số bất kỳ). Vậy `r` tần suất nhỏ nhất cũng khả thi. Tham lam lấy các tần suất nhỏ nhất tới khi không đủ ⇒ đạt `r` lớn nhất có thể.
4. **Dừng đúng lúc:** nếu `k < f_i` (tần suất thứ `i` sau sắp) thì mọi `f_j` với `j ≥ i` đều `≥ f_i > k` ⇒ không xóa sạch được thêm giá trị nào.

## 11. Phân tích Time / Space

Gọi `n = len(arr)`, `d` = số giá trị phân biệt (`d ≤ n`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi tập k chỉ số | `O(C(n, k) · n)` | `O(n)` |
| 2. Counter + sort | `O(n + d log d)` | `O(d)` |
| 3. Min-heap | `O(n + r log d)` (`r` giá trị bị xóa) | `O(d)` |
| 4. Counting sort tần suất | `O(n)` | `O(n)` |

Trường hợp xấu cho Cách 2: mọi phần tử khác nhau (`d = n`). Cách 4 luôn tuyến tính.

## 12. Bộ test

Ví dụ đề, edge case (`k = 0`, `k = n`, mọi phần tử giống nhau, mọi phần tử khác nhau, tần suất bằng nhau), đối chiếu brute force trên mảng nhỏ ngẫu nhiên (seed cố định), đối chiếu chéo các cách nhanh trên mảng lớn.

In [20]:
FAST = [least_unique_sort, least_unique_heap, least_unique_bucket,
        lambda a, k: Solution().findLeastNumOfUniqueInts(a, k)]

def check(arr, k, want=None, use_brute=True):
    if want is None:
        want = least_unique_brute(arr, k) if use_brute else FAST[0](arr, k)
    for f in FAST:
        assert f(list(arr), k) == want, (arr, k, f(list(arr), k), want)
    return want

for arr, k, want in examples:
    assert least_unique_brute(arr, k) == want
    check(arr, k, want)

assert check([1, 2, 3], 0) == 3                  # k = 0
assert check([1, 2, 3], 3) == 0                  # k = n
assert check([7, 7, 7, 7], 3) == 1               # một giá trị, không xóa sạch được
assert check([7, 7, 7, 7], 4) == 0
assert check([1, 2, 3, 4, 5], 2) == 3            # mọi phần tử khác nhau
assert check([1, 1, 2, 2, 3, 3], 3) == 2         # tần suất bằng nhau, dư 1 lượt
assert check([10**9, 1, 10**9], 1) == 1
print("Ví dụ + edge case: OK")

Ví dụ + edge case: OK


In [21]:
rng = random.Random(2022)
for _ in range(1500):
    n = rng.randint(1, 11)
    arr = [rng.randint(1, rng.choice([2, 4, 10])) for _ in range(n)]
    k = rng.randint(0, n)
    check(arr, k)
for _ in range(300):
    n = rng.randint(1, 2000)
    arr = [rng.randint(1, rng.choice([5, 100, 10**9])) for _ in range(n)]
    check(arr, rng.randint(0, n), use_brute=False)
print("Random: 1500 test so brute + 300 test so chéo: OK")

Random: 1500 test so brute + 300 test so chéo: OK


In [22]:
# Ràng buộc lớn
rng = random.Random(6)
big = [rng.randint(1, 10**9) for _ in range(10**5)]
big2 = [rng.randint(1, 300) for _ in range(10**5)]
for label, a in (("d≈n", big), ("d=300", big2)):
    for k in (0, 5 * 10**4, 10**5):
        t = time.perf_counter()
        r = Solution().findLeastNumOfUniqueInts(a, k)
        dt = time.perf_counter() - t
        assert r == least_unique_bucket(a, k)
        print(f"{label}, k={k}: {r}, {dt:.3f}s")

d≈n, k=0: 99995, 0.009s
d≈n, k=50000: 49995, 0.010s
d≈n, k=100000: 0, 0.012s
d=300, k=0: 300, 0.003s
d=300, k=50000: 144, 0.003s
d=300, k=100000: 0, 0.003s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Xóa giá trị xuất hiện **nhiều nhất** trước

In [23]:
def wrong_largest_first(arr, k):
    freqs = sorted(Counter(arr).values(), reverse=True)   # SAI: phải tăng dần
    for i, f in enumerate(freqs):
        if k < f:
            return len(freqs) - i
        k -= f
    return 0

arr, k = [1, 1, 1, 2, 3], 3
print("SAI :", wrong_largest_first(arr, k), "| đúng:", least_unique_sort(arr, k))

SAI : 2 | đúng: 1


Dùng 3 lượt để xóa sạch số `1` chỉ giảm được 1 giá trị; dùng 2 lượt xóa `2` và `3` giảm được 2. **Sửa:** sắp tần suất **tăng dần**.

### Lỗi 2 — Trừ `k` mà không kiểm tra đủ (k âm vẫn tính là xóa sạch)

In [24]:
def wrong_overspend(arr, k):
    freqs = sorted(Counter(arr).values())
    removed = 0
    for f in freqs:
        if k <= 0:
            break
        k -= f                                # SAI: có thể làm k âm
        removed += 1
    return len(freqs) - removed

arr, k, want = examples[1]
print("SAI :", wrong_overspend(arr, k), "| đúng:", want)

SAI : 1 | đúng: 2


Với `k = 1` còn lại và tần suất `2`, bản sai vẫn "xóa sạch" (k thành `−1`). **Sửa:** chỉ xóa khi `k >= f`.

### Lỗi 3 — So sánh `k > f` thay vì `k >= f`

In [25]:
def wrong_strict(arr, k):
    freqs = sorted(Counter(arr).values())
    for i, f in enumerate(freqs):
        if k > f:                             # SAI: k == f vẫn xóa sạch được
            k -= f
        else:
            return len(freqs) - i
    return 0

arr, k, want = examples[0]
print("SAI :", wrong_strict(arr, k), "| đúng:", want)

SAI : 2 | đúng: 1


**Sửa:** `if k >= f` (hoặc `if k < f: return ...`).

### Lỗi 4 — Nghĩ mỗi lượt xóa giảm một giá trị: `len(set(arr)) − k`

In [26]:
def wrong_subtract(arr, k):
    return len(set(arr)) - k                  # SAI

for arr, k, want in [examples[1], ([1, 1, 1], 2, 1)]:
    print("SAI :", wrong_subtract(arr, k), "| đúng:", least_unique_sort(arr, k), "| arr =", arr, "k =", k)

SAI : 1 | đúng: 2 | arr = [4, 3, 1, 1, 3, 3, 2] k = 3
SAI : -1 | đúng: 1 | arr = [1, 1, 1] k = 2


Bỏ qua việc một giá trị có thể xuất hiện nhiều lần, thậm chí ra số âm. **Sửa:** làm việc trên tần suất.

### Lỗi 5 — Xóa `k` phần tử **nhỏ nhất về giá trị**

Nhầm "ít xuất hiện" với "giá trị nhỏ".

In [27]:
def wrong_smallest_values(arr, k):
    return len(set(sorted(arr)[k:]))          # SAI: bỏ k giá trị nhỏ nhất

arr, k = [1, 1, 1, 5, 9], 2
print("SAI :", wrong_smallest_values(arr, k), "| đúng:", least_unique_sort(arr, k))
print("Bản sai bỏ hai số 1 (nhỏ nhất) -> còn {1, 5, 9}; đúng ra bỏ 5 và 9 -> chỉ còn {1}.")

SAI : 3 | đúng: 1
Bản sai bỏ hai số 1 (nhỏ nhất) -> còn {1, 5, 9}; đúng ra bỏ 5 và 9 -> chỉ còn {1}.


**Sửa:** sắp theo **tần suất**, không theo giá trị. (Một cách đúng khác: sắp `arr` theo khóa `(cnt[x], x)` rồi bỏ `k` phần tử đầu — xem bài tập 4.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Ít / nhiều giá trị phân biệt nhất sau khi xóa / đổi k phần tử" ⇒ **đếm tần suất** trước.
- "Tối đa số món chọn được với ngân sách `k`" ⇒ **tham lam rẻ nhất trước** (sort tăng hoặc min-heap).
- Tần suất bị chặn bởi `n` ⇒ có thể **counting sort** để đạt `O(n)`.
- Luôn tự hỏi phần **dư** của ngân sách đi đâu — ở đây vô hại vì `k ≤ n`.
- Dùng exchange argument để thuyết phục bản thân tham lam đúng; kiểm chứng bằng brute force nhỏ.

## 15. Biến thể của bài

1. **Nhiều giá trị phân biệt nhất sau khi xóa đúng k phần tử:** xóa các bản sao thừa trước; đáp án `min(d, n − k)` (bài tập 3).
2. **Đáp án cho mọi `k` từ `0` tới `n`:** prefix sum trên tần suất đã sắp (bài tập 5).
3. **1338 — Reduce Array Size to The Half:** xóa sạch ít giá trị nhất để mảng giảm còn một nửa — tham lam **nhiều nhất trước** (ngược chiều!).
4. **Xóa tối đa k phần tử** (thay vì đúng k): đáp án y hệt, vì xóa thêm không làm tăng số giá trị.
5. **Mỗi giá trị có trọng số chi phí khác nhau** (không phải tần suất): vẫn là "tối đa số món với ngân sách" ⇒ sort theo chi phí.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Thử mọi tập `k` chỉ số | `O(C(n,k)·n)` | `O(n)` | Kiểm chứng `n` nhỏ |
| 2. Counter + sort | Tham lam tần suất nhỏ trước | `O(n + d log d)` | `O(d)` | Mặc định, ngắn gọn |
| 3. Min-heap | Pop tần suất nhỏ nhất khi còn đủ `k` | `O(n + r log d)` | `O(d)` | `k` nhỏ, ít phần tử bị xóa |
| 4. Counting sort | `fc[f]` số giá trị có tần suất `f` | `O(n)` | `O(n)` | Muốn tuyến tính tuyệt đối |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `findLeastNumOfUniqueInts([2,4,1,8,3,5,1,3], 3)` = ?

**Bài 2 (tính tay):** Với `arr = [1,1,1,2,2,3,4,4,4,4]`, `k = 4`, viết tần suất đã sắp và từng bước tham lam.

**Bài 3 (điền TODO):** Viết `most_unique_after_removals(arr, k)`: số giá trị phân biệt **nhiều nhất** có thể còn lại sau khi xóa đúng `k` phần tử.

**Bài 4 (tự viết lại):** Viết lại lời giải theo kiểu **sắp `arr` theo khóa `(cnt[x], x)`** rồi bỏ `k` phần tử đầu, đếm số giá trị phân biệt còn lại.

**Bài 5 (mini challenge):** Viết `answers_for_all_k(arr)` trả về list `res` độ dài `n + 1`, `res[k]` là đáp án bài gốc với `k`, tổng thời gian `O(n log n)`.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def most_unique_after_removals(arr: List[int], k: int) -> int:
    # TODO: xóa bản sao thừa trước (n - d phần tử), rồi mới phải xóa giá trị
    raise NotImplementedError

# Bài 4 — TODO
def least_unique_by_key(arr: List[int], k: int) -> int:
    # TODO: cnt = Counter(arr); sorted(arr, key=...)[k:]
    raise NotImplementedError

# Bài 5 — TODO
def answers_for_all_k(arr: List[int]) -> List[int]:
    # TODO: prefix sum của tần suất đã sắp; với mỗi k, số giá trị xóa sạch = số prefix <= k
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Đếm: bốn giá trị xuất hiện 1 lần, hai giá trị xuất hiện 2 lần. (2) `k = 3` xóa sạch ba giá trị xuất hiện 1 lần. (3) `6 − 3`.

**Bài 2.** (1) Tần suất `[1, 2, 3, 4]`. (2) Xóa `1` (k = 3), xóa `2` (k = 1). (3) `3 > 1` ⇒ dừng, còn 2 giá trị.

**Bài 3.** (1) Có `n − d` bản sao thừa. (2) Nếu `k ≤ n − d` thì vẫn còn đủ `d` giá trị. (3) Ngược lại mỗi lượt xóa thêm mất 1 giá trị: `d − (k − (n − d)) = n − k`. Gộp: `min(d, n − k)`.

**Bài 4.** (1) `cnt = Counter(arr)`. (2) `sorted(arr, key=lambda x: (cnt[x], x))` — khóa `x` để các bản sao của cùng giá trị đứng liền nhau. (3) `len(set(...[k:]))`.

**Bài 5.** (1) `freqs` sắp tăng, `pre[i]` = tổng `i` tần suất đầu. (2) Với `k`, số giá trị xóa sạch = số `i ≥ 1` có `pre[i] ≤ k` (dùng `bisect_right(pre, k) − 1`). (3) Hoặc quét `k` tăng dần với con trỏ.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1, Bài 2
r1 = least_unique_sort([2, 4, 1, 8, 3, 5, 1, 3], 3)
print("Bài 1:", r1)
assert r1 == 3
arr2 = [1, 1, 1, 2, 2, 3, 4, 4, 4, 4]
print("Bài 2: tần suất sắp =", sorted(Counter(arr2).values()), "| đáp án =", least_unique_sort(arr2, 4))
assert sorted(Counter(arr2).values()) == [1, 2, 3, 4] and least_unique_sort(arr2, 4) == 2

Bài 1: 3
Bài 2: tần suất sắp = [1, 2, 3, 4] | đáp án = 2


In [30]:
from bisect import bisect_right

def most_unique_ans(arr, k):
    return min(len(set(arr)), len(arr) - k)

def least_unique_by_key_ans(arr, k):
    cnt = Counter(arr)
    return len(set(sorted(arr, key=lambda x: (cnt[x], x))[k:]))

def most_unique_brute(arr, k):
    n = len(arr)
    return max(len({arr[i] for i in range(n) if i not in set(c)}) for c in combinations(range(n), k))

rng = random.Random(23)
for _ in range(800):
    arr = [rng.randint(1, 5) for _ in range(rng.randint(1, 10))]
    k = rng.randint(0, len(arr))
    assert most_unique_ans(arr, k) == most_unique_brute(arr, k)
    assert least_unique_by_key_ans(arr, k) == least_unique_brute(arr, k)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [31]:
def answers_for_all_k_ans(arr):
    freqs = sorted(Counter(arr).values())
    pre = [0]
    for f in freqs:
        pre.append(pre[-1] + f)
    d = len(freqs)
    return [d - (bisect_right(pre, k) - 1) for k in range(len(arr) + 1)]

rng = random.Random(29)
for _ in range(300):
    arr = [rng.randint(1, 8) for _ in range(rng.randint(1, 40))]
    assert answers_for_all_k_ans(arr) == [least_unique_sort(arr, k) for k in range(len(arr) + 1)]
print("Bài 5:", answers_for_all_k_ans([4, 3, 1, 1, 3, 3, 2]), "| OK 300 test")

Bài 5: [4, 3, 2, 2, 1, 1, 1, 0] | OK 300 test


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "tối đa số món với ngân sách k":**

```python
freqs = sorted(Counter(arr).values())   # chi phí tăng dần
for i, f in enumerate(freqs):
    if k < f:
        return len(freqs) - i           # các món từ i trở đi không mua được
    k -= f
return 0
```

**Checklist:**
- [ ] Làm việc trên **tần suất**, không phải giá trị.
- [ ] Sắp tần suất **tăng dần** (rẻ nhất trước).
- [ ] Điều kiện xóa sạch là `k >= f` (có dấu bằng).
- [ ] Không để `k` âm; dừng ngay khi không đủ.
- [ ] Trả `0` khi xóa sạch tất cả (`k = n`).
- [ ] Đã thử `k = 0`, `k = n`, mọi phần tử giống nhau / khác nhau.